In [10]:
import json
import networkx as nx
import dash
import dash_cytoscape as cyto
from dash import html, Input, Output
from networkx.drawing.nx_agraph import graphviz_layout



# === Load graph from JSON ===
with open("mutuals_graph.json") as f:
    adj_dict = json.load(f)

# Build directed graph and include all nodes
G = nx.DiGraph()
all_nodes = set(adj_dict.keys()) | {v for neighbors in adj_dict.values() for v in neighbors}
G.add_nodes_from(all_nodes)  # ensure all users are in the graph

for u, neighbors in adj_dict.items():
    for v in neighbors:
        G.add_edge(u, v)


# Identify mutual edges
mutual_pairs = set()
for u, v in G.edges():
    if G.has_edge(v, u):
        mutual_pairs.add(tuple(sorted((u, v))))

# Generate Cytoscape elements
positions = graphviz_layout(G, prog="sfdp")  # great for large graphs
elements = []

for node in G.nodes():
    scale = 6  # Try values between 3–10 for looser spacing
    x, y = positions[node]
    elements.append({
        'data': {'id': node, 'label': node},
        'position': {'x': x * scale, 'y': y * scale},
        'selectable': True,
        'grabbable': False
    })


for u, v in G.edges():
    pair = tuple(sorted((u, v)))
    if pair in mutual_pairs:
        elements.append({'data': {'source': u, 'target': v}, 'classes': 'mutual'})
    else:
        elements.append({'data': {'source': u, 'target': v}, 'classes': 'oneway'})

# === Create Dash App ===
app = dash.Dash(__name__)
app.layout = html.Div([
    cyto.Cytoscape(
        id='cytoscape',
        layout={'name': 'preset'},
        style={'width': '100%', 'height': '800px'},
        elements=elements,
        stylesheet=[
            {'selector': 'node', 'style': {
                'background-color': '#888',
                'label': 'data(label)',
                'font-size': '10px'
            }},
            {'selector': '.mutual', 'style': {
                'line-color': 'grey',
                'width': 2
            }},
            {'selector': '.oneway', 'style': {
                'line-color': 'green',
                'width': 2,
                'target-arrow-shape': 'triangle',
                'target-arrow-color': 'green',
                'arrow-scale': 0.8
            }},
            {'selector': '.highlight', 'style': {
                'background-color': 'red',
                'line-color': 'red',
                'width': 3
            }},
        ]
    )
])

# === Click callback to highlight neighbors ===
@app.callback(
    Output('cytoscape', 'elements'),
    Input('cytoscape', 'tapNodeData')
)
def highlight_neighbors(clicked_node):
    if not clicked_node:
        return elements

    node_id = clicked_node['id']
    neighbors = set(G.successors(node_id)) | set(G.predecessors(node_id)) | {node_id}

    updated_elements = []
    for el in elements:
        if 'source' in el['data']:  # edge
            updated_elements.append(el)
        else:
            nid = el['data']['id']
            new_el = el.copy()
            new_el['classes'] = el.get('classes', '')
            if nid in neighbors:
                new_el['classes'] += ' highlight'
            updated_elements.append(new_el)
    return updated_elements

# Launch in browser
app.run_server(debug=True)


In [11]:
import networkx as nx
import numpy as np
import pandas as pd

# Basic Graph Analysis

# Number of nodes and edges
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# In-degree and out-degree
in_degrees = [deg for _, deg in G.in_degree()]
out_degrees = [deg for _, deg in G.out_degree()]

avg_in_degree = np.mean(in_degrees)
avg_out_degree = np.mean(out_degrees)
max_in_degree = np.max(in_degrees)
max_out_degree = np.max(out_degrees)
min_in_degree = np.min(in_degrees)
min_out_degree = np.min(out_degrees)

# Weakly connected components
num_connected_components = nx.number_weakly_connected_components(G)
largest_cc = max(nx.weakly_connected_components(G), key=len)
subgraph_largest_cc = G.subgraph(largest_cc)

# Display results
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")
print(f"Average in-degree: {avg_in_degree:.2f}")
print(f"Average out-degree: {avg_out_degree:.2f}")
print(f"Max in-degree: {max_in_degree}, Min in-degree: {min_in_degree}")
print(f"Max out-degree: {max_out_degree}, Min out-degree: {min_out_degree}")
print(f"Number of weakly connected components: {num_connected_components}")
print(f"Size of largest connected component: {len(largest_cc)}")


Number of nodes: 660
Number of edges: 19140
Average in-degree: 29.00
Average out-degree: 29.00
Max in-degree: 115, Min in-degree: 0
Max out-degree: 141, Min out-degree: 0
Number of weakly connected components: 6
Size of largest connected component: 655


In [12]:
import networkx as nx
from torch_geometric.utils import to_networkx


# Compute in-degrees and out-degrees
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {node: in_degrees[node] + out_degrees[node] for node in G.nodes()}
# Sort by total degree
top_total = sorted(total_degrees.items(), key=lambda x: x[1], reverse=True)


print("Top 10 nodes by total-degree:")
for node, deg in top_total[:10]:
    print(f"Node {node}: {in_degrees[node]} incoming edges and {out_degrees[node]} outgoing edges")

print("\nBottom 10 nodes by total-degree:")
for node, deg in top_total[-1:-11:-1]:
    print(f"Node {node}: {in_degrees[node]} incoming edges and {out_degrees[node]} outgoing edges")






Top 10 nodes by total-degree:
Node itsthejukeofyork: 115 incoming edges and 141 outgoing edges
Node willbrodner: 108 incoming edges and 127 outgoing edges
Node ruopuj22: 107 incoming edges and 125 outgoing edges
Node greilly16: 107 incoming edges and 122 outgoing edges
Node grant__mak: 108 incoming edges and 118 outgoing edges
Node wyatt_jernigan: 102 incoming edges and 120 outgoing edges
Node imalittleman_ish: 98 incoming edges and 112 outgoing edges
Node jasonbrovich: 98 incoming edges and 111 outgoing edges
Node jferrante21: 87 incoming edges and 116 outgoing edges
Node winstonyau99: 89 incoming edges and 106 outgoing edges

Bottom 10 nodes by total-degree:
Node oliviamckinley_: 0 incoming edges and 0 outgoing edges
Node tiffiniez: 0 incoming edges and 0 outgoing edges
Node _spinsta_666: 0 incoming edges and 0 outgoing edges
Node baejaeyun7353: 0 incoming edges and 0 outgoing edges
Node yujiasun31: 0 incoming edges and 0 outgoing edges
Node cookingwithklaassen: 1 incoming edges and 